In [0]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
from dateutil import parser
import re

print("✓ Core libraries imported")

In [0]:
%run /Workspace/data_project/1_setup/utilities

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://sportx-bar/{data_source}'
landing_path = f"{base_path}/landing/"
processed_path = f"{base_path}/processed/"
print("Base Path: ", base_path)
print("Landing Path: ", landing_path)
print("Processed Path: ", processed_path)


# define the tables
# Create data directory
os.makedirs("data", exist_ok=True)
print("\n✓ Data directory ready")

In [0]:
# Read ALL files from S3 landing zone (Full Load)
print(f"Reading ALL files from: {landing_path}")

try:
    # Read CSV from S3 using Spark (UC credentials)
    df_spark = spark.read.csv(f"{landing_path}/*.csv", header=True, inferSchema=True)
    
    # Convert to pandas
    df = df_spark.toPandas()
    
    # Add Bronze metadata
    df["read_timestamp"] = pd.Timestamp.now()
    df["file_name"] = f"{data_source}.csv"
    df["file_size"] = None
    
    print(f"\n✓ Loaded {len(df)} rows from landing zone")
    print(f"\nColumns: {list(df.columns)}")
    print(f"\nFirst 5 rows:")
    display(df.head(5))
    
except Exception as e:
    print(f"\n⚠️  No files found: {landing_path}")
    print(f"\nTo run this pipeline:")
    print(f"  1. Upload CSV files to: {landing_path}")
    print(f"  2. Re-run this notebook")
    df = None

In [0]:
# Stop if no data
if df is None:
    print("\n⚠️  No data to process. Stopping.")
    dbutils.notebook.exit("No data to process")

# Save Bronze layer (Full Load = overwrite mode)
bronze_path = f"data/bronze_{data_source}.csv"
df.to_csv(bronze_path, index=False)

print(f"\n✓ Bronze layer saved: {bronze_path}")
print(f"  Rows: {len(df)}")
print(f"  Columns: {len(df.columns)}")

In [0]:
# Move processed files from landing to processed folder
print(f"\nMoving files: {landing_path} → {processed_path}")

try:
    files = dbutils.fs.ls(landing_path)
    moved_count = 0
    
    for file_info in files:
        if file_info.name.endswith('.csv'):
            dbutils.fs.mv(
                file_info.path,
                f"{processed_path}{file_info.name}",
                True
            )
            moved_count += 1
            print(f"  ✓ Moved: {file_info.name}")
    
    print(f"\n✓ Moved {moved_count} files to processed folder")
    
except Exception as e:
    print(f"\n⚠️  Error moving files: {e}")
    print("  Files may have already been moved or path doesn't exist")

In [0]:
# Load Bronze data
df_orders = pd.read_csv(f"data/bronze_{data_source}.csv")
print(f"Loaded {len(df_orders)} rows from Bronze")
print(f"\nFirst 2 rows:")
display(df_orders.head(2))

In [0]:
# 1. Keep only rows where order_qty is present
print(f"Rows before filter: {len(df_orders)}")
df_orders = df_orders[df_orders['order_qty'].notna()]
print(f"Rows after removing null order_qty: {len(df_orders)}")


# 2. Clean customer_id → keep numeric, else set to 999999
def clean_customer_id(val):
    if pd.isna(val):
        return "999999"
    val_str = str(val)
    if val_str.replace('.', '', 1).isdigit():  # Handle integers stored as floats
        return str(int(float(val_str)))
    if val_str.isdigit():
        return val_str
    return "999999"

df_orders['customer_id'] = df_orders['customer_id'].apply(clean_customer_id)

# 3. Remove weekday name from date text
df_orders['order_placement_date'] = df_orders['order_placement_date'].astype(str).apply(
    lambda x: re.sub(r'^[A-Za-z]+,\s*', '', x)
)

# 4. Parse order_placement_date using multiple formats
def parse_date(date_str):
    if pd.isna(date_str) or date_str == 'None' or date_str == '':
        return None
    formats = ['%Y/%m/%d', '%d-%m-%Y', '%d/%m/%Y', '%B %d, %Y']
    for fmt in formats:
        try:
            return pd.to_datetime(date_str, format=fmt)
        except:
            continue
    try:
        return parser.parse(date_str)
    except:
        return None

df_orders['order_placement_date'] = df_orders['order_placement_date'].apply(parse_date)

# 5. Drop duplicates
print(f"\nRows before dedup: {len(df_orders)}")
df_orders = df_orders.drop_duplicates(
    subset=['order_id', 'order_placement_date', 'customer_id', 'product_id', 'order_qty'],
    keep='first'
)
print(f"Rows after dedup: {len(df_orders)}")

# 6. Convert product_id to string
df_orders['product_id'] = df_orders['product_id'].astype(str)

print(f"\n✓ Transformations complete")

In [0]:
# Check date range
min_date = df_orders['order_placement_date'].min()
max_date = df_orders['order_placement_date'].max()

print(f"\nDate range:")
print(f"  Min date: {min_date}")
print(f"  Max date: {max_date}")

In [0]:
# Load products dimension
df_products = pd.read_csv("data/silver_products.csv")
print(f"Loaded {len(df_products)} products")

# Join orders with products to get product_code
df_joined = df_orders.merge(
    df_products[['product_id', 'product_code']],
    on='product_id',
    how='inner'
)

print(f"\n✓ Joined with products: {len(df_joined)} rows")
print(f"\nFirst 5 rows:")
display(df_joined.head(5))

In [0]:
# Save Silver layer (Full Load = overwrite mode)
silver_path = f"data/silver_{data_source}.csv"
df_joined.to_csv(silver_path, index=False)

print(f"\n✓ Silver layer saved: {silver_path}")
print(f"  Rows: {len(df_joined)}")
print(f"  Columns: {len(df_joined.columns)}")

In [0]:
# Create Gold fact table (daily level)
df_gold = df_joined[[
    'order_id',
    'order_placement_date',
    'customer_id',
    'product_code',
    'product_id',
    'order_qty'
]].copy()

df_gold = df_gold.rename(columns={
    'order_placement_date': 'date',
    'customer_id': 'customer_code',
    'order_qty': 'sold_quantity'
})

print(f"\nGold fact table preview:")
display(df_gold.head(2))

In [0]:
# Save Gold daily fact (Full Load = overwrite mode)
gold_sb_path = f"data/gold_sb_fact_{data_source}.csv"
df_gold.to_csv(gold_sb_path, index=False)

print(f"\n✓ Gold daily fact saved: {gold_sb_path}")
print(f"  Rows: {len(df_gold)}")
print(f"  Columns: {len(df_gold.columns)}")

In [0]:
# Load Gold daily data for monthly aggregation
df_child = pd.read_csv(f"data/gold_sb_fact_{data_source}.csv")
df_child['date'] = pd.to_datetime(df_child['date'])

print(f"\nLoaded {len(df_child)} daily rows for aggregation")
print(f"\nFirst 10 rows:")
display(df_child.head(10))

In [0]:
print(f"Total daily rows: {len(df_child)}")

In [0]:
# Aggregate daily to monthly level
# Get first day of month
df_child['month_start'] = df_child['date'].dt.to_period('M').dt.to_timestamp()

# Group by month, product_code, customer_code
df_monthly = df_child.groupby(
    ['month_start', 'product_code', 'customer_code']
).agg({
    'sold_quantity': 'sum'
}).reset_index()

# Rename month_start to date
df_monthly = df_monthly.rename(columns={'month_start': 'date'})

print(f"\nMonthly aggregated rows: {len(df_monthly)}")
print(f"\nFirst 5 rows:")
display(df_monthly.head(5))

In [0]:
print(f"Monthly fact rows: {len(df_monthly)}")

In [0]:
# Save parent monthly fact (Full Load = overwrite mode)
gold_parent_path = f"data/gold_fact_{data_source}.csv"
df_monthly.to_csv(gold_parent_path, index=False)

print(f"\n✓ Parent monthly fact saved: {gold_parent_path}")
print(f"  Rows: {len(df_monthly)}")
print(f"  Columns: {len(df_monthly.columns)}")

print("\n" + "="*60)
print("FULL LOAD COMPLETE")
print("="*60)
print(f"Bronze: data/bronze_{data_source}.csv")
print(f"Silver: data/silver_{data_source}.csv")
print(f"Gold daily: data/gold_sb_fact_{data_source}.csv")
print(f"Gold monthly: data/gold_fact_{data_source}.csv")